# 实验 5：MMDFND 复现（中文微博多模态假新闻检测）

**前提**：`data 3/` 已在 Google Drive（含 CSV + 预处理好的 pkl）

**运行前**：Runtime → Change runtime type → **GPU (T4 或更高)**

In [ ]:
# ============================================================
# Step 0: 检查 GPU
# ============================================================
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: 需要 GPU！请在 Runtime → Change runtime type 选 GPU')

In [ ]:
# ============================================================
# Step 1: 挂载 Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
# 确认 data 3 存在
DATA3 = '/content/drive/MyDrive/data 3'
PROJECT = '/content/drive/MyDrive/fakenews-detector'
print(f'data 3: {os.path.isdir(DATA3)}')
print(f'project: {os.path.isdir(PROJECT)}')

# 列出 data 3 内容
for f in sorted(os.listdir(DATA3)):
    size = os.path.getsize(os.path.join(DATA3, f)) / 1e6
    print(f'  {f:30s} {size:>10.1f} MB')

In [ ]:
# ============================================================
# Step 2: 复制项目代码到 Colab 本地磁盘（Drive I/O 太慢）
# ============================================================
import shutil, time

LOCAL_PROJECT = '/content/fakenews-detector'

if not os.path.exists(LOCAL_PROJECT):
    print('复制代码到本地...')
    t0 = time.time()
    def ignore_fn(d, files):
        skip = set()
        for f in files:
            full = os.path.join(d, f)
            if os.path.isfile(full) and os.path.getsize(full) > 200_000_000:
                skip.add(f)
            if f in ('.git', '__pycache__', 'outputs', 'data 3'):
                skip.add(f)
        return skip
    shutil.copytree(PROJECT, LOCAL_PROJECT, ignore=ignore_fn)
    print(f'复制完成 ({time.time()-t0:.1f}s)')
else:
    print(f'已存在: {LOCAL_PROJECT}')

MMDFND_DIR = os.path.join(LOCAL_PROJECT, 'MMDFND')
print(f'MMDFND 目录: {MMDFND_DIR}')
print(f'  run.py: {os.path.exists(os.path.join(MMDFND_DIR, "run.py"))}')
print(f'  main.py: {os.path.exists(os.path.join(MMDFND_DIR, "main.py"))}')

In [ ]:
# ============================================================
# Step 3: 安装依赖
# ============================================================
!pip install -q transformers timm positional_encodings
!pip install -q cn_clip
print('依赖安装完成')

In [ ]:
# ============================================================
# Step 4: 下载 Chinese RoBERTa 预训练模型（如果还没有）
#         这是 MMDFND 的文本编码器，约 400MB
# ============================================================
import os

BERT_DIR = os.path.join(MMDFND_DIR, 'pretrained_model', 'chinese_roberta_wwm_base_ext_pytorch')

if os.path.exists(os.path.join(BERT_DIR, 'config.json')):
    print(f'Chinese RoBERTa 已存在: {BERT_DIR}')
else:
    print('下载 Chinese RoBERTa (hfl/chinese-roberta-wwm-ext)...')
    os.makedirs(BERT_DIR, exist_ok=True)
    from transformers import BertTokenizer, AutoModel
    tok = BertTokenizer.from_pretrained('hfl/chinese-roberta-wwm-ext')
    model = AutoModel.from_pretrained('hfl/chinese-roberta-wwm-ext')
    tok.save_pretrained(BERT_DIR)
    model.save_pretrained(BERT_DIR)
    print(f'已保存到 {BERT_DIR}')

    # MMDFND 的 BertTokenizer 需要 vocab.txt；如果 save_pretrained 没生成就手动下载
    vocab_path = os.path.join(BERT_DIR, 'vocab.txt')
    if not os.path.exists(vocab_path):
        print('vocab.txt 未自动生成，从 HuggingFace 手动下载...')
        import urllib.request
        urllib.request.urlretrieve(
            'https://huggingface.co/hfl/chinese-roberta-wwm-ext/resolve/main/vocab.txt',
            vocab_path,
        )
        print(f'已下载 vocab.txt ({os.path.getsize(vocab_path)/1e3:.0f} KB)')

for f in os.listdir(BERT_DIR):
    print(f'  {f}')

In [ ]:
# ============================================================
# Step 5: 链接 data 3 的 pkl 和 CSV 到 MMDFND/data/
#         pkl 太大（9GB+），用符号链接避免复制
# ============================================================
import os

MMDFND_DATA = os.path.join(MMDFND_DIR, 'data')
os.makedirs(MMDFND_DATA, exist_ok=True)

DATA3 = '/content/drive/MyDrive/data 3'

files_to_link = [
    'train_origin.csv', 'val_origin.csv', 'test_origin.csv',
    'train_loader.pkl', 'train_clip_loader.pkl',
    'val_loader.pkl', 'val_clip_loader.pkl',
    'test_loader.pkl', 'test_clip_loader.pkl',
]

for fname in files_to_link:
    src = os.path.join(DATA3, fname)
    dst = os.path.join(MMDFND_DATA, fname)
    if os.path.exists(dst):
        print(f'  OK  {fname}')
    elif os.path.exists(src):
        os.symlink(src, dst)
        print(f'  Linked {fname}')
    else:
        print(f'  MISSING {src}')

print(f'\nMMDFND/data/ 内容:')
for f in sorted(os.listdir(MMDFND_DATA)):
    p = os.path.join(MMDFND_DATA, f)
    real = os.path.realpath(p)
    size = os.path.getsize(real) / 1e6 if os.path.isfile(real) else 0
    print(f'  {f:30s} {size:>10.1f} MB')

In [ ]:
# ============================================================
# Step 6: 检查一切就绪
# ============================================================
import os

checks = {
    'MMDFND/main.py':      os.path.join(MMDFND_DIR, 'main.py'),
    'MMDFND/run.py':       os.path.join(MMDFND_DIR, 'run.py'),
    'Chinese RoBERTa':     os.path.join(BERT_DIR, 'config.json'),
    'train CSV':           os.path.join(MMDFND_DATA, 'train_origin.csv'),
    'train_loader.pkl':    os.path.join(MMDFND_DATA, 'train_loader.pkl'),
    'train_clip_loader':   os.path.join(MMDFND_DATA, 'train_clip_loader.pkl'),
    'val_loader.pkl':      os.path.join(MMDFND_DATA, 'val_loader.pkl'),
    'test_loader.pkl':     os.path.join(MMDFND_DATA, 'test_loader.pkl'),
}

all_ok = True
for name, path in checks.items():
    exists = os.path.exists(path)
    if not exists:
        all_ok = False
    print(f"  {'OK' if exists else 'MISSING':7s} {name}")

if all_ok:
    print('\n✅ 所有文件就绪，可以开始训练！')
else:
    print('\n❌ 有文件缺失，请检查上面的步骤')

In [ ]:
# ============================================================
# Step 7: 开始训练 MMDFND
#         --dataset weibo  对应 data 3 的 CSV 格式
#         --root_path      指向 MMDFND/data/（已链接 pkl）
#         --epoch 50       训练 50 轮（约 2-5 小时，看 GPU）
# ============================================================
%cd {MMDFND_DIR}

!python main.py \
    --dataset weibo \
    --root_path "./data/" \
    --epoch 50 \
    --batchsize 64 \
    --lr 0.0001 \
    --gpu 0

In [ ]:
# ============================================================
# Step 8: 查看结果 & 保存模型到 Drive
# ============================================================
import os, shutil

# 检查保存的模型
param_dir = os.path.join(MMDFND_DIR, 'param_model', 'MMDFND')
if os.path.isdir(param_dir):
    for f in os.listdir(param_dir):
        fp = os.path.join(param_dir, f)
        print(f'  {f} ({os.path.getsize(fp)/1e6:.1f} MB)')

    # 保存到 Drive
    drive_save = '/content/drive/MyDrive/fakenews-detector/MMDFND/param_model'
    os.makedirs(drive_save, exist_ok=True)
    for f in os.listdir(param_dir):
        src = os.path.join(param_dir, f)
        dst = os.path.join(drive_save, f)
        shutil.copy2(src, dst)
    print(f'\n模型已保存到 Drive: {drive_save}')
else:
    print('模型目录不存在，训练可能未完成')

## 常见问题

**Q: `cn_clip` import 报错?**
```
!pip install cn_clip
```
如果仍然报错，可能是 Chinese-CLIP 权重文件缺失。MMDFND 的 `clip_dataloader.py` 会尝试加载 `ViT-B-16` 模型。可以在 Step 7 之前运行：
```python
import cn_clip.clip as clip
clip.load_from_name('ViT-B-16', device='cpu', download_root=MMDFND_DIR)
```

**Q: `mae_pretrain_vit_base.pth` 缺失?**

MMDFND 模型代码里可能引用了 MAE 权重。如果报错，需要下载：
```bash
!wget -q https://dl.fbaipublicfiles.com/mae/pretrain/mae_pretrain_vit_base.pth -O {MMDFND_DIR}/mae_pretrain_vit_base.pth
```

**Q: 训练太慢?**
- 减小 `--batchsize 32`
- 减少 `--epoch 20`
- 确认用的是 GPU（`nvidia-smi` 查看）

**Q: OOM (显存不足)?**
- `--batchsize 16` 或 `--batchsize 8`
- T4 (16GB) 通常 batchsize=32 可行